In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Read the two files
final_hcris_v1996 = pd.read_csv('data/output/HCRIS_Data_v1996.txt', sep='\t')
final_hcris_v2010 = pd.read_csv('data/output/HCRIS_Data_v2010.txt', sep='\t')

# Create missing variables for columns introduced in v2010
final_hcris_v1996 = final_hcris_v1996.assign(
    hvbp_payment=pd.NA,
    hrrp_payment=pd.NA
)

In [3]:
# Combine v1996 and v2010 HCRIS forms
final_hcris = pd.concat([final_hcris_v1996, final_hcris_v2010], ignore_index=True)

# Convert date columns
for col in ['fy_end', 'fy_start', 'date_processed', 'date_created']:
    final_hcris[col] = pd.to_datetime(final_hcris[col], errors='coerce')

# Apply absolute value transformations
final_hcris['tot_discounts'] = pd.to_numeric(final_hcris['tot_discounts'], errors='coerce').abs()
final_hcris['hrrp_payment'] = pd.to_numeric(final_hcris['hrrp_payment'], errors='coerce').abs()

# Create fiscal year from fy_end
final_hcris['fyear'] = final_hcris['fy_end'].dt.year

# Sort by provider_number and fyear
final_hcris = final_hcris.sort_values(['provider_number', 'fyear'])

# Drop the original year column
final_hcris = final_hcris.drop(columns=['year'])

# Count hospitals/provider_number by year
count_by_year = final_hcris.groupby('fyear').size().reset_index(name='n')

print(count_by_year)
final_hcris.to_csv("final_hcris.csv", index=False)

/tmp/ipykernel_1437565/961390745.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_hcris = pd.concat([final_hcris_v1996, final_hcris_v2010], ignore_index=True)


    fyear     n
0    2008    23
1    2009  3539
2    2010  2634
3    2011  5860
4    2012  6231
5    2013  6159
6    2014  6168
7    2015  6154
8    2016  6188
9    2017  6176
10   2018  6136
11   2019  2612


In [4]:
# create count of reports by hospital fiscal year
final_hcris = final_hcris.copy()
final_hcris["total_reports"] = (
    final_hcris.groupby(["provider_number", "fyear"])["provider_number"]
    .transform("size")
)

# create running total of reports
final_hcris["report_number"] = (
    final_hcris.groupby(["provider_number", "fyear"])
    .cumcount() + 1

)

# identify hospitals with only one report per fiscal year
# this will be the first set of hospitals in the final dataset
unique_hcris1 = (
    final_hcris.loc[final_hcris["total_reports"] == 1]
    .drop(columns=["report", "total_reports", "report_number", "npi", "status"], errors="ignore")
    .assign(source="unique reports")
    .copy()
)

# identify hospitals with multiple reports per fiscal year
duplicate_hcris = final_hcris.loc[final_hcris["total_reports"] > 1].copy()

# calculate elapsed time between fy start and fy end for hospitals with multiple reports
duplicate_hcris["time_diff"] = duplicate_hcris["fy_end"] - duplicate_hcris["fy_start"]

In [5]:
# if you want plain integer days like R often uses downstream
duplicate_hcris["time_diff_days"] = duplicate_hcris["time_diff"].dt.days
duplicate_hcris["total_days"] = (
    duplicate_hcris.groupby(["provider_number", "fyear"])["time_diff_days"]
    .transform("sum")
)

# if the elapsed time within a fy sums to around 365, then just take the total of the two
# this will be the second set of hospitals in the final dataset
duplicate_hcris_for_sum = duplicate_hcris.copy()
duplicate_hcris_for_sum["hrrp_payment"] = duplicate_hcris_for_sum["hrrp_payment"].fillna(0)
duplicate_hcris_for_sum["hvbp_payment"] = duplicate_hcris_for_sum["hvbp_payment"].fillna(0)

unique_hcris2 = (
    duplicate_hcris_for_sum.loc[duplicate_hcris_for_sum["total_days"] < 370]
    .groupby(["provider_number", "fyear"], as_index=False)
    .agg({
        "beds": "max",
        "tot_charges": "sum",
        "tot_discounts": "sum",
        "tot_operating_exp": "sum",
        "ip_charges": "sum",
        "icu_charges": "sum",
        "ancillary_charges": "sum",
        "tot_discharges": "sum",
        "mcare_discharges": "sum",
        "mcaid_discharges": "sum",
        "tot_mcare_payment": "sum",
        "secondary_mcare_payment": "sum",
        "hvbp_payment": "sum",
        "hrrp_payment": "sum",
        "fy_start": "min",
        "fy_end": "max",
        "date_processed": "max",
        "date_created": "min",
        "street": "first",
        "city": "first",
        "state": "first",
        "zip": "first",
        "county": "first",
    })
)

In [6]:
unique_hcris2["source"] = "total for year"

# identify hospitals with more than one report and with elapsed time exceeding 370 days
duplicate_hcris2 = duplicate_hcris.loc[duplicate_hcris["total_days"] >= 370].copy()

duplicate_hcris2["max_days"] = (
    duplicate_hcris2.groupby(["provider_number", "fyear"])["time_diff_days"]
    .transform("max")
)
duplicate_hcris2["max_date"] = (
    duplicate_hcris2.groupby(["provider_number","fyear"])["fy_end"]
    .transform("max")
)

In [7]:
# identify hospitals with one report (out of multiple) that appears to cover the full year
# this will be the third set of hospitals in the final dataset
unique_hcris3 = (
    duplicate_hcris2.loc[
        (duplicate_hcris2["max_days"] == duplicate_hcris2["time_diff_days"]) &
        (duplicate_hcris2["time_diff_days"] > 360) &
        (duplicate_hcris2["max_date"] == duplicate_hcris2["fy_end"])
    ]
    .drop(
        columns=[
            "report", "total_reports", "report_number", "npi", "status", "max_days", "time_diff", "time_diff_days", "total_days", "max_date"
    ],
    errors="ignore"
        )
    .assign(source="primary report")
    .copy()
)

In [8]:
# identify remaining hospitals
# anti_join(duplicate_hcris2, unique_hcris3, by=c("provider_number", "fyear"))

keys_to_remove = unique_hcris3[["provider_number", "fyear"]].drop_duplicates()

duplicate_hcris3 = duplicate_hcris2.merge(
    keys_to_remove,
    on=["provider_number", "fyear"],
    how="left",
    indicator=True
)

duplicate_hcris3 = (
    duplicate_hcris3.loc[duplicate_hcris3["_merge"] == "left_only"]
    .drop(columns=["_merge"])
    .copy()
)

In [9]:
# weight selected variables by time_diff / total_days
weighted_cols = [
    "tot_charges", "tot_discounts", "tot_operating_exp", "ip_charges", "icu_charges","ancillary_charges", "tot_discharges", "mcare_discharges","mcaid_discharges", "tot_mcare_payment", "secondary_mcare_payment", "hvbp_payment", "hrrp_payment"
]
weight = duplicate_hcris3["time_diff_days"] / duplicate_hcris3["total_days"]
duplicate_hcris3[weighted_cols] = duplicate_hcris3[weighted_cols].multiply(weight, axis=0)

In [10]:
# form weighted average of values for each fiscal year
duplicate_hcris3["hrrp_payment"] = duplicate_hcris3["hrrp_payment"].fillna(0)
duplicate_hcris3["hvbp_payment"] = duplicate_hcris3["hvbp_payment"].fillna(0)

unique_hcris4 = (
duplicate_hcris3.groupby(["provider_number", "fyear"], as_index=False)
.agg({
        "beds": "max",
        "tot_charges": "sum",
        "tot_discounts": "sum",
        "tot_operating_exp": "sum",
        "ip_charges": "sum",
        "icu_charges": "sum",
        "ancillary_charges": "sum",
        "tot_discharges": "sum",
        "mcare_discharges": "sum",
        "mcaid_discharges": "sum",
        "tot_mcare_payment": "sum",
        "secondary_mcare_payment": "sum",
        "hvbp_payment": "sum",
        "hrrp_payment": "sum",
        "fy_start": "min",
        "fy_end": "max",
        "date_processed": "max",
        "date_created": "min",
        "street": "first",
        "city": "first",
        "state": "first",
        "zip": "first",
        "county": "first",
    })
)
unique_hcris4["source"] = "weighted_average"

In [11]:
# Combine all datasets
final_hcris_data = pd.concat(
    [unique_hcris1, unique_hcris2, unique_hcris3, unique_hcris4],
    ignore_index=True
)

# Rename and sort
final_hcris_data = (
    final_hcris_data
    .rename(columns={"fyear": "year"})
    .sort_values(["provider_number", "year"])
)

# Ensure output folder exists
output_dir = Path("data/output")
output_dir.mkdir(parents=True, exist_ok=True)

# Write one CSV per year
years = sorted(final_hcris_data["year"].dropna().unique())
for y in years:
    (
    final_hcris_data.loc[final_hcris_data["year"] == y]
        .to_csv(output_dir / f"data-{int(y)}.csv", index=False)
    )
    
# Print summary (equivalent to cat in R)
print("Years written:", years)
print("Total rows:", len(final_hcris_data))

Years written: [np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019)]
Total rows: 57115
